# Week 65, Day 2: Learning an HMM: Baum-Welch and model selection

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w065/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import poisson
g0 = np.random.default_rng(652)
T = 1200
A_true = np.array([[0.95, 0.05], [0.02, 0.98]])
rates_true = np.array([1.5, 6.0])
z = np.zeros(T, int)
for t in range(1, T):
    z[t] = g0.choice(2, p=A_true[z[t - 1]])
x = g0.poisson(rates_true[z])

def forward_backward(B, A, pi):
    T_, K = B.shape
    al = np.zeros((T_, K)); c = np.zeros(T_); be = np.ones((T_, K))
    a = pi * B[0]; c[0] = a.sum(); al[0] = a / c[0]
    for t in range(1, T_):
        a = B[t] * (al[t - 1] @ A); c[t] = a.sum(); al[t] = a / c[t]
    for t in range(T_ - 2, -1, -1):
        be[t] = (A @ (B[t + 1] * be[t + 1])) / c[t + 1]
    g = al * be; g /= g.sum(1, keepdims=True)
    return g, al, be, c
print("ready: 1,200 bins of a two-state Poisson HMM with unknown parameters")

### Exercise: Baum-Welch for a Poisson HMM

Complete the E step (pairwise marginals xi) and the M step (transition matrix, initial distribution and Poisson rates) of baum_welch, and fit the two-state model from a rough starting guess.

In [ ]:
def baum_welch(x, K, rates0, A0, pi0, iters=200, tol=1e-8):
    rates, A, pi = rates0.copy(), A0.copy(), pi0.copy()
    hist = []
    for it in range(iters):
        B = poisson.pmf(x[:, None], rates) + 1e-300
        g, al, be, c = forward_backward(B, A, pi)
        hist.append(np.sum(np.log(c)))
        xi = al[:-1, :, None] * A[None, :, :] * (B[1:] * be[1:])[:, None, :]
        xi /= ...                                        # normalise each time's K x K table
        A = ...                                          # expected transitions, row-normalised
        pi = g[0]
        rates = (g * x[:, None]).sum(0) / g.sum(0)
        if len(hist) > 1 and hist[-1] - hist[-2] < tol:
            break
    return rates, A, pi, hist, g

rates_hat, A_hat, pi_hat, ll_hist, gam = baum_welch(x, 2, np.array([x.mean() * 0.5, x.mean() * 1.5]), np.array([[0.9, 0.1], [0.1, 0.9]]), np.array([0.5, 0.5]))
order = np.argsort(rates_hat)
print(f"{len(ll_hist)} iterations; log-likelihood {ll_hist[0]:.1f} -> {ll_hist[-1]:.1f}")
print(f"rates {np.round(rates_hat[order], 3)} (true {rates_true})")
print("transition matrix:\n", np.round(A_hat[np.ix_(order, order)], 3), "\ntrue:\n", A_true)

In [ ]:
import numpy as np
def test_monotone():
    "the log-likelihood never decreases"
    assert all(b >= a - 1e-8 for a, b in zip(ll_hist, ll_hist[1:]))
def test_recover():
    "the learned rates and transition probabilities are close to the truth"
    assert np.allclose(np.sort(rates_hat), np.sort(rates_true), rtol=0.15)
    assert abs(A_hat[0, 0] - 0.95) < 0.08 and abs(A_hat[1, 1] - 0.98) < 0.05

test_monotone()
test_recover()
print("All checks passed")

### Exercise: Many starts, many optima

Fit a three-state model from 12 random starts and record the final log-likelihoods. Keep the run with the highest final log-likelihood, and report how many distinct optima appear (to within 0.1 nats) and that run's rates.

In [ ]:
gs = np.random.default_rng(6521)
finals, fits = [], []
for s in range(12):
    r0 = gs.uniform(0.5, 10, 3)
    A0 = gs.uniform(size=(3, 3)) + 3 * np.eye(3); A0 /= A0.sum(1, keepdims=True)
    out = baum_welch(x, 3, r0, A0, np.full(3, 1 / 3), iters=80)
    finals.append(out[3][-1]); fits.append(out)
finals = np.array(finals)
best = fits[int(...)]
best_rates = np.sort(best[0])
distinct = len(np.unique(np.round(finals, 1)))
print(f"final log-likelihoods from {len(finals)} starts: best {finals.max():.2f}, worst {finals.min():.2f}, {distinct} distinct values")
print(f"best run's rates {np.round(best_rates, 2)}")

In [ ]:
import numpy as np
def test_spread():
    "random starts do not all reach the same log-likelihood"
    assert max(finals) - min(finals) > 1.0
def test_best():
    "the best run keeps a high-rate state near 6 and splits the low-rate state"
    assert abs(best_rates[-1] - 6.0) < 1.0 and best_rates[0] < 3.0

test_spread()
test_best()
print("All checks passed")

### Exercise: Held-out likelihood and a simulation check

Fit models with 1 to 4 states on the first 800 bins (best of 3 starts each) and evaluate the log-likelihood per bin of the last 400. Then simulate 5,000 bins from the two-state fit and compare the distribution of dwell times in the high-rate state with the data's.

In [ ]:
xtr, xte = x[:800], x[800:]
heldout = {}
for K in (1, 2, 3, 4):
    runs = []
    for s in range(3):
        r0 = gs.uniform(0.5, 10, K)
        A0 = gs.uniform(size=(K, K)) + 3 * np.eye(K); A0 /= A0.sum(1, keepdims=True)
        runs.append(baum_welch(xtr, K, r0, A0, np.full(K, 1 / K), iters=80))
    fit = max(runs, key=lambda o: o[3][-1])
    B_te = poisson.pmf(xte[:, None], fit[0]) + 1e-300
    _, _, _, c_te = forward_backward(B_te, fit[1], fit[2])
    heldout[K] = ...
    if K == 2:
        fit2 = fit
print("held-out log-likelihood per bin:", {k: round(float(v), 4) for k, v in heldout.items()})

def dwell_times(states, k):
    runs, n = [], 0
    for s_ in states:
        if s_ == k: n += 1
        elif n: runs.append(n); n = 0
    return np.array(runs + ([n] if n else []))

hi = int(np.argmax(fit2[0]))
gsim = np.random.default_rng(3)
zs = np.zeros(5000, int)
for t_ in range(1, 5000):
    zs[t_] = gsim.choice(2, p=fit2[1][zs[t_ - 1]])
dwell_sim = dwell_times(zs, hi)
dwell_true = dwell_times(z, 1)
print(f"median dwell in the high-rate state: data {np.median(dwell_true):.0f} bins, simulated from the fit {np.median(dwell_sim):.0f} bins")

In [ ]:
import numpy as np
def test_two():
    "held-out log-likelihood per bin is highest for two states, or within 0.01 of it"
    assert heldout[2] >= max(heldout.values()) - 0.01
def test_dwell():
    "dwell times simulated from the fit are of the same order as the data's"
    assert 0.5 < np.median(dwell_sim) / np.median(dwell_true) < 2

test_two()
test_dwell()
print("All checks passed")